# LeetCode 773 — Sliding Puzzle bằng Python

- Link: https://leetcode.com/problems/sliding-puzzle/
- Độ khó: **Hard**
- Chủ đề: BFS trên không gian trạng thái, BFS hai chiều, A* (heuristic Manhattan), IDDFS, Tính chẵn lẻ của hoán vị

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def slidingPuzzle(self, board: List[List[int]]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Trò chơi xếp hình trượt 2×3: có 5 viên gạch đánh số `1..5` và một ô trống `0`. Mỗi bước, trượt một viên gạch **kề** ô trống vào ô trống. Hỏi cần **ít nhất** bao nhiêu bước để đạt trạng thái `[[1,2,3],[4,5,0]]`?

Đây là bài mẫu mực của **tìm kiếm trên không gian trạng thái**: mỗi cách sắp bảng là một **đỉnh**, mỗi bước trượt là một **cạnh**, và "ít bước nhất" là **đường đi ngắn nhất trong đồ thị không trọng số** → BFS. Bảng nhỏ (chỉ `6! = 720` trạng thái) nên mọi kỹ thuật đều chạy được, giúp ta so sánh BFS, BFS hai chiều, A*, IDDFS.

Sau notebook này bạn sẽ:

1. Mã hoá bảng thành **chuỗi** và dựng **bảng láng giềng** của ô trống.
2. Viết **BFS** chuẩn với `deque` và tập `visited`.
3. Cài **BFS hai chiều** và **A\*** với heuristic **khoảng cách Manhattan**.
4. Dùng **IDDFS** (tìm sâu dần) làm baseline tiết kiệm bộ nhớ.
5. Hiểu **tính chẵn lẻ của hoán vị** để biết ngay khi nào trả `-1`.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Bảng `board` kích thước `2 × 3` chứa các số `0..5`, mỗi số đúng một lần (`0` là ô trống). Một **bước**: chọn một ô **kề cạnh** (trên/dưới/trái/phải) với ô `0` và đổi chỗ nó với `0`. Trạng thái đích là `[[1,2,3],[4,5,0]]`. Trả về số bước **ít nhất** để đạt đích; nếu không thể, trả `-1`.

**Ví dụ 1**

```text
Input: board = [[1,2,3],[4,0,5]]
Output: 1
Giải thích: đổi 0 và 5.
```

**Ví dụ 2**

```text
Input: board = [[1,2,3],[5,4,0]]
Output: -1
Giải thích: không thể giải.
```

**Ví dụ 3**

```text
Input: board = [[4,1,2],[5,0,3]]
Output: 5
Giải thích: một lời giải tối ưu:
[[4,1,2],[5,0,3]] -> [[4,1,2],[0,5,3]] -> [[0,1,2],[4,5,3]] -> [[1,0,2],[4,5,3]]
-> [[1,2,0],[4,5,3]] -> [[1,2,3],[4,5,0]]
```

**Ràng buộc**

- `board.length == 2`, `board[i].length == 3`
- `0 <= board[i][j] <= 5`
- Mỗi giá trị trong `board` là **duy nhất**.

In [1]:
from typing import List

EXAMPLES = [
    ([[1, 2, 3], [4, 0, 5]], 1),
    ([[1, 2, 3], [5, 4, 0]], -1),
    ([[4, 1, 2], [5, 0, 3]], 5),
    ([[3, 2, 4], [1, 5, 0]], 14),
    ([[1, 2, 3], [4, 5, 0]], 0),
]
TARGET = "123450"

def encode(board):
    return ''.join(str(x) for row in board for x in row)

print([encode(b) for b, _ in EXAMPLES])

['123405', '123540', '412503', '324150', '123450']


## 3. Kiến thức nền

### 3.1. Mã hoá bảng thành chuỗi, chỉ số ↔ toạ độ

Duỗi bảng theo hàng: ô `(r, c)` ↔ chỉ số `i = 3·r + c`; ngược lại `r, c = divmod(i, 3)`. Chuỗi bất biến nên dùng làm khoá `set`/`dict` được.

```text
chỉ số:  0 1 2
         3 4 5
```

In [2]:
for i in range(6):
    print(i, "->", divmod(i, 3))
print(encode([[4, 1, 2], [5, 0, 3]]))

0 -> (0, 0)
1 -> (0, 1)
2 -> (0, 2)
3 -> (1, 0)
4 -> (1, 1)
5 -> (1, 2)
412503


### 3.2. Bảng láng giềng của ô trống

Từ vị trí `i` của `0`, các vị trí có thể đổi chỗ:

```text
0: [1, 3]     1: [0, 2, 4]     2: [1, 5]
3: [0, 4]     4: [1, 3, 5]     5: [2, 4]
```

Có thể viết tay hoặc sinh tự động từ toạ độ.

In [3]:
def build_neighbors(R=2, C=3):
    nb = []
    for i in range(R * C):
        r, c = divmod(i, C)
        lst = []
        for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            rr, cc = r + dr, c + dc
            if 0 <= rr < R and 0 <= cc < C:
                lst.append(rr * C + cc)
        nb.append(sorted(lst))
    return nb

NEI = build_neighbors()
print(NEI)

[[1, 3], [0, 2, 4], [1, 5], [0, 4], [1, 3, 5], [2, 4]]


### 3.3. Đổi chỗ hai ký tự trong chuỗi

Chuỗi bất biến → đổi sang list, đổi chỗ, ghép lại.

In [4]:
def swap(s, i, j):
    lst = list(s)
    lst[i], lst[j] = lst[j], lst[i]
    return ''.join(lst)

def next_states(s):
    z = s.index('0')
    return [swap(s, z, j) for j in NEI[z]]

print(next_states("412503"))

['402513', '412053', '412530']


### 3.4. BFS cho đường đi ngắn nhất trên đồ thị không trọng số

BFS duyệt theo **lớp**: mọi đỉnh cách nguồn `d` bước được lấy ra trước mọi đỉnh cách `d + 1`. Vì vậy lần đầu chạm đích chính là khoảng cách ngắn nhất. Đánh dấu `visited` **khi đẩy vào hàng đợi** để mỗi đỉnh vào hàng đợi một lần.

In [5]:
from collections import deque

def bfs_dist(start, nbrs):
    dist = {start: 0}
    q = deque([start])
    while q:
        u = q.popleft()
        for v in nbrs(u):
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return dist

# Ví dụ nhỏ: đồ thị đường thẳng 0-1-2-3
print(bfs_dist(0, lambda x: [y for y in (x - 1, x + 1) if 0 <= y <= 3]))

{0: 0, 1: 1, 2: 2, 3: 3}


### 3.5. Heap cho A\*

A\* lấy ra trạng thái có `f = g + h` nhỏ nhất (g = số bước đã đi, h = ước lượng còn lại) → dùng `heapq`.

In [6]:
import heapq
h = []
for item in [(5, "x"), (2, "y"), (4, "z")]:
    heapq.heappush(h, item)
print(heapq.heappop(h), heapq.heappop(h))

(2, 'y') (4, 'z')


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Không gian trạng thái

Có `6! = 720` cách sắp 6 số. Mỗi trạng thái có 2 hoặc 3 láng giềng (tuỳ vị trí ô trống). Ta tìm đường đi ngắn nhất từ trạng thái đầu tới `"123450"`.

### 4.2. Tính tay ví dụ 1

`"123405"`: `0` ở chỉ số 4, láng giềng `[1, 3, 5]` → các trạng thái `"103425"`, `"123045"`, `"123450"`. Trạng thái thứ ba là đích → 1 bước.

### 4.3. Các lớp BFS cho ví dụ 3 (`"412503"`)

- Lớp 0: `412503`
- Lớp 1: `402513`, `412053`, `412530`
- Lớp 2: ..., `012453` (từ `412053`), ...
- Lớp 3: `102453`
- Lớp 4: `120453`
- Lớp 5: `123450` ✔

### 4.4. Một nửa số trạng thái không tới được

Chỉ **360** trong 720 trạng thái tới được đích. Ta sẽ chứng minh bằng **tính chẵn lẻ** ở mục 6 và kiểm chứng bằng BFS.

In [7]:
all_from_target = bfs_dist(TARGET, next_states)
print("số trạng thái tới được:", len(all_from_target), "/ 720")
print("khoảng cách xa nhất:", max(all_from_target.values()))

số trạng thái tới được: 360 / 720
khoảng cách xa nhất: 21


## 5. Cách 1 — Baseline: IDDFS (tìm kiếm sâu dần)

**Ý tưởng.** DFS thuần có thể đi lạc rất sâu và không cho đường ngắn nhất. **IDDFS**: chạy DFS giới hạn độ sâu `L = 0, 1, 2, ...`; lần đầu tìm thấy đích ở giới hạn `L` thì `L` là đáp án. Để tránh vòng lặp, không đi ngược lại nước vừa đi và không đi lại trạng thái trên đường hiện tại.

Ta biết (từ BFS ở trên) khoảng cách lớn nhất là 21, nên nếu tới `L = 21` vẫn không thấy thì trả `-1`. (Trong lời giải "thật" không biết trước số này, ta sẽ dùng kiểm tra chẵn lẻ.)

In [8]:
def iddfs(board, max_depth=21):
    start = encode(board)
    if start == TARGET:
        return 0

    def dfs(s, z, prev_z, depth, limit, path):
        if s == TARGET:
            return True
        if depth == limit:
            return False
        for j in NEI[z]:
            if j == prev_z:                  # không đi ngược nước vừa đi
                continue
            t = swap(s, z, j)
            if t in path:
                continue
            path.add(t)
            if dfs(t, j, z, depth + 1, limit, path):
                return True
            path.remove(t)
        return False

    z0 = start.index('0')
    for limit in range(1, max_depth + 1):
        if dfs(start, z0, -1, 0, limit, {start}):
            return limit
    return -1

for b, exp in EXAMPLES:
    print(b, "->", iddfs(b), "(kỳ vọng", exp, ")")

[[1, 2, 3], [4, 0, 5]] -> 1 (kỳ vọng 1 )
[[1, 2, 3], [5, 4, 0]] -> -1 (kỳ vọng -1 )
[[4, 1, 2], [5, 0, 3]] -> 5 (kỳ vọng 5 )
[[3, 2, 4], [1, 5, 0]] -> 14 (kỳ vọng 14 )
[[1, 2, 3], [4, 5, 0]] -> 0 (kỳ vọng 0 )


**Độ phức tạp.** Với hệ số nhánh hiệu dụng `β` (≈ 1.5 sau khi bỏ nước đi ngược) và đáp án `d`, IDDFS duyệt `O(β^d)` nút, bộ nhớ chỉ `O(d)`. Với trạng thái **không giải được**, phải duyệt tới giới hạn tối đa — tốn nhất.

**Nút thắt.** IDDFS lặp lại công việc ở các giới hạn nhỏ và có thể thăm một trạng thái nhiều lần theo các đường khác nhau. Không gian chỉ có 720 trạng thái → nhớ hết bằng `visited` và chạy **BFS**, mỗi trạng thái đúng một lần.

## 6. Bước đột phá

**Insight 1 — BFS trên trạng thái chuỗi.** Mỗi trạng thái thăm một lần → tối đa 720 trạng thái, mỗi trạng thái ≤ 3 cạnh. Gần như tức thì.

**Insight 2 — Tìm từ hai phía.** Cùng lúc mở rộng từ trạng thái đầu và từ đích; gặp nhau ở giữa. Số trạng thái thăm giảm từ `β^d` xuống `2·β^(d/2)`.

**Insight 3 — Heuristic.** Mỗi viên gạch cần ít nhất `|Δr| + |Δc|` bước để về chỗ; mỗi bước chỉ di chuyển một viên → `h = Σ khoảng cách Manhattan` (không tính ô trống) **không bao giờ ước lượng quá** → A\* với `h` này cho đường ngắn nhất.

**Insight 4 — Tính chẵn lẻ (khi nào `-1`).** Đọc các số khác 0 theo hàng, đếm **số nghịch thế** (cặp `i < j` mà `a[i] > a[j]`).

- Trượt ngang: thứ tự đọc **không đổi** → nghịch thế không đổi.
- Trượt dọc (bảng rộng 3): viên gạch nhảy qua đúng **2** viên khác trong thứ tự đọc → nghịch thế thay đổi `±2` hoặc `0` → **chẵn lẻ không đổi**.

Đích `12345` có 0 nghịch thế (chẵn). Vậy trạng thái có **số nghịch thế lẻ** không bao giờ giải được. (Điều ngược lại — chẵn thì luôn giải được — được BFS kiểm chứng: đúng 360 trạng thái chẵn, tất cả tới được.)

**Quy tắc nếu… thì…**

- Nếu số nghịch thế lẻ → trả `-1` ngay.
- Nếu cần đáp án nhiều lần → BFS **một lần từ đích**, tra bảng.
- Nếu không gian lớn (ví dụ 3×3 hay 4×4) → BFS hai chiều hoặc A\*/IDA\*.

In [9]:
def inversions(s):
    a = [c for c in s if c != '0']
    return sum(1 for i in range(len(a)) for j in range(i + 1, len(a)) if a[i] > a[j])

from itertools import permutations
even = [''.join(p) for p in permutations("012345") if inversions(''.join(p)) % 2 == 0]
print("số trạng thái nghịch thế chẵn:", len(even))
print("trùng đúng tập tới được:", set(even) == set(all_from_target))

số trạng thái nghịch thế chẵn: 360
trùng đúng tập tới được: True


## 7. Các cách tiếp theo

### 7.1. Cách 2 — BFS từ trạng thái đầu

**Giải thích.** Hàng đợi chứa `(trạng thái, số bước)`, `visited` chứa các trạng thái đã đẩy vào. Lấy ra, sinh láng giềng; gặp đích thì trả.

In [10]:
def bfs(board):
    start = encode(board)
    if start == TARGET:
        return 0
    seen = {start}
    q = deque([(start, 0)])
    while q:
        s, d = q.popleft()
        z = s.index('0')
        for j in NEI[z]:
            t = swap(s, z, j)
            if t == TARGET:
                return d + 1
            if t not in seen:
                seen.add(t)
                q.append((t, d + 1))
    return -1

for b, exp in EXAMPLES:
    print(bfs(b), exp)

1 1
-1 -1
5 5
14 14
0 0


**Code nộp LeetCode (Cách 2).**

In [11]:
class SolutionBFS:
    def slidingPuzzle(self, board: List[List[int]]) -> int:
        from collections import deque
        nei = [[1, 3], [0, 2, 4], [1, 5], [0, 4], [1, 3, 5], [2, 4]]
        target = "123450"
        start = ''.join(str(x) for row in board for x in row)
        if start == target:
            return 0
        seen = {start}
        q = deque([(start, 0)])
        while q:
            s, d = q.popleft()
            z = s.index('0')
            for j in nei[z]:
                lst = list(s)
                lst[z], lst[j] = lst[j], lst[z]
                t = ''.join(lst)
                if t == target:
                    return d + 1
                if t not in seen:
                    seen.add(t)
                    q.append((t, d + 1))
        return -1

print([SolutionBFS().slidingPuzzle(b) for b, _ in EXAMPLES])

[1, -1, 5, 14, 0]


**Độ phức tạp.** `V = 6! = 720` trạng thái (tổng quát `(RC)!`), mỗi trạng thái tạo chuỗi dài `RC` cho ≤ 4 láng giềng → `O((RC)! · RC)`. Bộ nhớ `O((RC)! · RC)`.

### 7.2. Cách 3 — BFS hai chiều

**Giải thích.** Giữ hai "biên" `front` (từ đầu) và `back` (từ đích), mỗi bên là một `set`. Mỗi vòng, mở rộng **biên nhỏ hơn** thêm một lớp; nếu một trạng thái mới nằm trong biên kia → đáp án = số lớp đã mở rộng tổng cộng. Cần `visited` chung để không quay lại.

In [12]:
def bfs_bidirectional(board):
    start = encode(board)
    if start == TARGET:
        return 0
    if inversions(start) % 2:
        return -1
    front, back = {start}, {TARGET}
    visited = {start, TARGET}
    steps = 0
    while front and back:
        if len(front) > len(back):
            front, back = back, front           # luôn mở rộng phía nhỏ hơn
        steps += 1
        nxt = set()
        for s in front:
            z = s.index('0')
            for j in NEI[z]:
                t = swap(s, z, j)
                if t in back:
                    return steps
                if t not in visited:
                    visited.add(t)
                    nxt.add(t)
        front = nxt
    return -1

for b, exp in EXAMPLES:
    print(bfs_bidirectional(b), exp)

1 1
-1 -1
5 5
14 14
0 0


**Lưu ý tinh tế:** dùng chung một `visited` cho cả hai phía vẫn đúng ở đây, vì khi một trạng thái mới đã được phía kia thăm, ta kiểm tra `t in back` **trước** — nhưng nếu `t` đã nằm trong phía kia ở một lớp **cũ hơn** (không còn trong `back`), ta có thể bỏ lỡ. Thực tế điều đó không xảy ra: nếu `t` được phía kia thăm ở lớp cũ, thì hai phía đã gặp nhau sớm hơn. Để chắc chắn, ta kiểm chứng trên **toàn bộ 720 trạng thái** ở mục 12.

**Code nộp LeetCode (Cách 3).**

In [13]:
class SolutionBidirectional:
    def slidingPuzzle(self, board: List[List[int]]) -> int:
        nei = [[1, 3], [0, 2, 4], [1, 5], [0, 4], [1, 3, 5], [2, 4]]
        target = "123450"
        start = ''.join(str(x) for row in board for x in row)
        if start == target:
            return 0
        front, back, visited = {start}, {target}, {start, target}
        steps = 0
        while front and back:
            if len(front) > len(back):
                front, back = back, front
            steps += 1
            nxt = set()
            for s in front:
                z = s.index('0')
                for j in nei[z]:
                    lst = list(s)
                    lst[z], lst[j] = lst[j], lst[z]
                    t = ''.join(lst)
                    if t in back:
                        return steps
                    if t not in visited:
                        visited.add(t)
                        nxt.add(t)
            front = nxt
        return -1

print([SolutionBidirectional().slidingPuzzle(b) for b, _ in EXAMPLES])

[1, -1, 5, 14, 0]


**Độ phức tạp.** Xấu nhất vẫn `O(V · RC)`, nhưng số trạng thái thăm thường giảm mạnh (≈ `2·β^(d/2)` thay vì `β^d`).

### 7.3. Cách 4 — A\* với heuristic Manhattan

**Giải thích.** Ưu tiên trạng thái có `f = g + h` nhỏ nhất, trong đó `g` = số bước đã đi, `h` = tổng khoảng cách Manhattan của các viên gạch (không tính `0`) tới vị trí đích. `h` **chấp nhận được** (không bao giờ lớn hơn số bước thật) và **nhất quán** (mỗi bước làm `h` đổi đúng ±1), nên lần đầu lấy đích ra khỏi heap là tối ưu, và mỗi trạng thái chỉ cần mở rộng một lần.

In [14]:
GOAL_POS = {ch: divmod(i, 3) for i, ch in enumerate(TARGET)}

def manhattan(s):
    total = 0
    for i, ch in enumerate(s):
        if ch != '0':
            r, c = divmod(i, 3)
            gr, gc = GOAL_POS[ch]
            total += abs(r - gr) + abs(c - gc)
    return total

def astar(board):
    start = encode(board)
    if inversions(start) % 2:
        return -1
    g = {start: 0}
    heap = [(manhattan(start), 0, start)]
    while heap:
        f, d, s = heapq.heappop(heap)
        if s == TARGET:
            return d
        if d > g[s]:
            continue                             # bản ghi cũ
        z = s.index('0')
        for j in NEI[z]:
            t = swap(s, z, j)
            nd = d + 1
            if nd < g.get(t, 1 << 30):
                g[t] = nd
                heapq.heappush(heap, (nd + manhattan(t), nd, t))
    return -1

print("h(412503) =", manhattan("412503"))
for b, exp in EXAMPLES:
    print(astar(b), exp)

h(412503) = 5
1 1
-1 -1
5 5
14 14
0 0


**Code nộp LeetCode (Cách 4).**

In [15]:
class SolutionAStar:
    def slidingPuzzle(self, board: List[List[int]]) -> int:
        import heapq
        nei = [[1, 3], [0, 2, 4], [1, 5], [0, 4], [1, 3, 5], [2, 4]]
        target = "123450"
        goal = {ch: divmod(i, 3) for i, ch in enumerate(target)}
        start = ''.join(str(x) for row in board for x in row)

        a = [c for c in start if c != '0']
        inv = sum(a[i] > a[j] for i in range(5) for j in range(i + 1, 5))
        if inv % 2:
            return -1

        def h(s):
            return sum(abs(i // 3 - goal[c][0]) + abs(i % 3 - goal[c][1])
                       for i, c in enumerate(s) if c != '0')

        best = {start: 0}
        heap = [(h(start), 0, start)]
        while heap:
            _, d, s = heapq.heappop(heap)
            if s == target:
                return d
            if d > best[s]:
                continue
            z = s.index('0')
            for j in nei[z]:
                lst = list(s)
                lst[z], lst[j] = lst[j], lst[z]
                t = ''.join(lst)
                if d + 1 < best.get(t, 99):
                    best[t] = d + 1
                    heapq.heappush(heap, (d + 1 + h(t), d + 1, t))
        return -1

print([SolutionAStar().slidingPuzzle(b) for b, _ in EXAMPLES])

[1, -1, 5, 14, 0]


**Độ phức tạp.** Xấu nhất `O(V log V · RC)`; thực tế thăm ít trạng thái hơn BFS nhiều vì heuristic hướng tìm kiếm về phía đích.

### 7.4. Cách 5 — Tiền xử lý: BFS một lần từ đích, tra bảng

**Giải thích.** Đồ thị vô hướng (mỗi bước trượt đảo ngược được) → khoảng cách từ `s` tới đích = khoảng cách từ đích tới `s`. Chạy BFS **một lần** từ `"123450"`, lưu `dist` cho cả 360 trạng thái tới được. Mỗi truy vấn chỉ là tra `dict` (không có → `-1`). Hữu ích khi phải trả lời **nhiều** bảng.

In [16]:
DIST_FROM_TARGET = bfs_dist(TARGET, next_states)

def table_lookup(board):
    return DIST_FROM_TARGET.get(encode(board), -1)

for b, exp in EXAMPLES:
    print(table_lookup(b), exp)

1 1
-1 -1
5 5
14 14
0 0


**Code nộp LeetCode (Cách 5)** — bảng tính một lần ở mức lớp (dùng chung giữa các test case).

In [17]:
class SolutionTable:
    _dist = None

    @classmethod
    def _build(cls):
        from collections import deque
        nei = [[1, 3], [0, 2, 4], [1, 5], [0, 4], [1, 3, 5], [2, 4]]
        dist = {"123450": 0}
        q = deque(["123450"])
        while q:
            s = q.popleft()
            z = s.index('0')
            for j in nei[z]:
                lst = list(s)
                lst[z], lst[j] = lst[j], lst[z]
                t = ''.join(lst)
                if t not in dist:
                    dist[t] = dist[s] + 1
                    q.append(t)
        cls._dist = dist

    def slidingPuzzle(self, board: List[List[int]]) -> int:
        if SolutionTable._dist is None:
            SolutionTable._build()
        return SolutionTable._dist.get(''.join(str(x) for row in board for x in row), -1)

print([SolutionTable().slidingPuzzle(b) for b, _ in EXAMPLES])

[1, -1, 5, 14, 0]


**Độ phức tạp.** Tiền xử lý `O(V · RC)` một lần; mỗi truy vấn `O(RC)`. Bộ nhớ `O(V · RC)`.

## 8. TRACE từng bước

Trace **Cách 2 (BFS)** trên ví dụ 3 `"412503"`: in từng lớp BFS và lưu cha để **khôi phục đường đi**.

In [18]:
def show_board(s):
    return f"[{s[0]} {s[1]} {s[2]} / {s[3]} {s[4]} {s[5]}]"

def trace_bfs(board):
    start = encode(board)
    parent = {start: None}
    layer = [start]
    d = 0
    while layer:
        print(f"lớp {d}: {layer}")
        if TARGET in layer:
            break
        nxt = []
        for s in layer:
            z = s.index('0')
            for j in NEI[z]:
                t = swap(s, z, j)
                if t not in parent:
                    parent[t] = s
                    nxt.append(t)
        layer, d = nxt, d + 1
    if TARGET not in parent:
        print("không tới được đích")
        return -1
    path, cur = [], TARGET
    while cur:
        path.append(cur); cur = parent[cur]
    path.reverse()
    print("đường đi:")
    for k, s in enumerate(path):
        print(f"  bước {k}: {show_board(s)}  (h = {manhattan(s)})")
    return len(path) - 1

trace_bfs([[4, 1, 2], [5, 0, 3]])

lớp 0: ['412503']
lớp 1: ['402513', '412053', '412530']
lớp 2: ['042513', '420513', '012453', '410532']
lớp 3: ['542013', '423510', '102453', '401532']
lớp 4: ['542103', '423501', '120453', '152403', '041532', '431502']
lớp 5: ['502143', '542130', '403521', '423051', '123450', '152043', '152430', '541032', '431052', '431520']
đường đi:
  bước 0: [4 1 2 / 5 0 3]  (h = 5)
  bước 1: [4 1 2 / 0 5 3]  (h = 4)
  bước 2: [0 1 2 / 4 5 3]  (h = 3)
  bước 3: [1 0 2 / 4 5 3]  (h = 2)
  bước 4: [1 2 0 / 4 5 3]  (h = 1)
  bước 5: [1 2 3 / 4 5 0]  (h = 0)


5

Bảng tóm tắt các lớp đầu:

| lớp | các trạng thái |
|---|---|
| 0 | `412503` |
| 1 | `402513`, `412053`, `412530` |
| 2 | 4 trạng thái mới, trong đó có `012453` |
| ... | ... |
| 5 | chứa `123450` |

Trace **Cách 4 (A\*)**: in thứ tự lấy ra khỏi heap với `g`, `h`, `f`.

In [19]:
def trace_astar(board, limit=20):
    start = encode(board)
    g = {start: 0}
    heap = [(manhattan(start), 0, start)]
    k = 0
    while heap:
        f, d, s = heapq.heappop(heap)
        if d > g[s]:
            continue
        k += 1
        print(f"lấy #{k:2}: {show_board(s)} g={d} h={manhattan(s)} f={f}")
        if s == TARGET or k >= limit:
            break
        z = s.index('0')
        for j in NEI[z]:
            t = swap(s, z, j)
            if d + 1 < g.get(t, 1 << 30):
                g[t] = d + 1
                heapq.heappush(heap, (d + 1 + manhattan(t), d + 1, t))
    print("số trạng thái A* mở rộng:", k, "| BFS sẽ thăm:",
          sum(1 for v in bfs_dist(start, next_states).values() if v <= 5))

trace_astar([[4, 1, 2], [5, 0, 3]])

lấy # 1: [4 1 2 / 5 0 3] g=0 h=5 f=5
lấy # 2: [4 1 2 / 0 5 3] g=1 h=4 f=5
lấy # 3: [0 1 2 / 4 5 3] g=2 h=3 f=5
lấy # 4: [1 0 2 / 4 5 3] g=3 h=2 f=5
lấy # 5: [1 2 0 / 4 5 3] g=4 h=1 f=5
lấy # 6: [1 2 3 / 4 5 0] g=5 h=0 f=5
số trạng thái A* mở rộng: 6 | BFS sẽ thăm: 28


## 9. Code nộp LeetCode

Bản chọn: **Cách 2 (BFS)** kèm kiểm tra chẵn lẻ để thoát sớm — chuẩn mực, dễ giải thích.

In [20]:
class Solution:
    def slidingPuzzle(self, board: List[List[int]]) -> int:
        from collections import deque
        # láng giềng của từng chỉ số trong bảng 2x3 duỗi thành chuỗi
        nei = [[1, 3], [0, 2, 4], [1, 5], [0, 4], [1, 3, 5], [2, 4]]
        target = "123450"
        start = ''.join(str(x) for row in board for x in row)
        if start == target:
            return 0
        # chẵn lẻ: số nghịch thế lẻ -> không bao giờ giải được
        a = [c for c in start if c != '0']
        if sum(a[i] > a[j] for i in range(5) for j in range(i + 1, 5)) % 2:
            return -1
        seen = {start}
        q = deque([(start, 0)])
        while q:
            s, d = q.popleft()
            z = s.index('0')                 # vị trí ô trống
            for j in nei[z]:
                lst = list(s)
                lst[z], lst[j] = lst[j], lst[z]
                t = ''.join(lst)
                if t == target:
                    return d + 1
                if t not in seen:
                    seen.add(t)              # đánh dấu khi đẩy vào hàng đợi
                    q.append((t, d + 1))
        return -1

for b, exp in EXAMPLES:
    assert Solution().slidingPuzzle(b) == exp
print("OK")

OK


Giải thích từng dòng:

- `nei`: bảng láng giềng viết tay (mục 3.2).
- `start`: mã hoá bảng thành chuỗi 6 ký tự.
- Kiểm tra chẵn lẻ: không bắt buộc (BFS cũng trả `-1` sau khi duyệt hết 360 trạng thái), nhưng giúp thoát sớm và thể hiện hiểu biết.
- `if t == target: return d + 1`: kiểm tra khi **sinh** trạng thái — sớm hơn một lớp so với kiểm tra khi lấy ra.
- `seen.add(t)` ngay khi đẩy: tránh đẩy trùng.

## 10. Vì sao thuật toán đúng

**BFS.** Bất biến: hàng đợi luôn chứa các trạng thái theo thứ tự khoảng cách không giảm, và chênh nhau tối đa 1. Khi trạng thái `t` được đẩy lần đầu với `d + 1`, không có đường nào ngắn hơn tới `t` (nếu có, `t` đã được đẩy từ một lớp trước). Vậy lần đầu chạm đích cho đáp án tối ưu. Nếu hàng đợi cạn mà không gặp đích → đích không tới được → `-1`.

**Chẵn lẻ.** Đã chứng minh ở mục 6: mỗi bước giữ nguyên chẵn lẻ số nghịch thế; đích có 0 nghịch thế. Do đó trạng thái lẻ không tới được. (Chiều ngược lại kiểm chứng bằng BFS trên toàn không gian.)

**A\*.** Heuristic Manhattan nhất quán: một bước di chuyển đúng một viên gạch một ô, nên `h` thay đổi đúng ±1 ≤ chi phí bước (1). Với heuristic nhất quán, A\* lấy mỗi trạng thái ra lần đầu với `g` tối ưu.

**BFS hai chiều.** Đồ thị vô hướng; gặp nhau lần đầu khi tổng số lớp hai phía nhỏ nhất → đúng khoảng cách (kiểm chứng toàn bộ ở mục 12).

## 11. Phân tích Time/Space

Gọi `V = (R·C)!` (= 720), `N = R·C` (= 6), `d` = đáp án.

| Cách | Time | Space |
|---|---|---|
| 1. IDDFS | `O(β^d)` (mũ theo đáp án) | `O(d)` |
| 2. BFS | `O(V · N)` | `O(V · N)` |
| 3. BFS hai chiều | `O(V · N)` xấu nhất, thường ≈ `β^(d/2)` | như trên |
| 4. A\* | `O(V log V · N)` xấu nhất, thực tế ít hơn | `O(V · N)` |
| 5. Tra bảng | `O(V · N)` một lần + `O(N)`/truy vấn | `O(V · N)` |

In [21]:
import time, random
all_states = [''.join(p) for p in permutations("012345")]
boards = [[[int(c) for c in s[:3]], [int(c) for c in s[3:]]] for s in all_states]
for name, f in [("bfs", bfs), ("bidir", bfs_bidirectional), ("astar", astar), ("table", table_lookup)]:
    t0 = time.perf_counter()
    for b in boards:
        f(b)
    print(f"{name:6} {1000 * (time.perf_counter() - t0):8.1f} ms cho 720 bảng")
random.seed(773)
sample = random.sample(boards, 40)
t0 = time.perf_counter()
for b in sample:
    iddfs(b)
print(f"iddfs  {1000 * (time.perf_counter() - t0):8.1f} ms cho 40 bảng ngẫu nhiên")

bfs       167.3 ms cho 720 bảng


bidir      23.0 ms cho 720 bảng


astar      62.0 ms cho 720 bảng
table       0.7 ms cho 720 bảng


iddfs     132.8 ms cho 40 bảng ngẫu nhiên


## 12. Bộ test

In [22]:
methods = {
    "bfs": bfs, "bidir": bfs_bidirectional, "astar": astar, "table": table_lookup,
    "SolutionBFS": lambda b: SolutionBFS().slidingPuzzle(b),
    "SolutionBidirectional": lambda b: SolutionBidirectional().slidingPuzzle(b),
    "SolutionAStar": lambda b: SolutionAStar().slidingPuzzle(b),
    "SolutionTable": lambda b: SolutionTable().slidingPuzzle(b),
    "Solution": lambda b: Solution().slidingPuzzle(b),
}

# 12.1 Ví dụ + edge cases
edge = EXAMPLES + [
    ([[1, 2, 3], [4, 5, 0]], 0),           # đã ở đích
    ([[1, 2, 0], [4, 5, 3]], 1),
    ([[2, 1, 3], [4, 5, 0]], -1),           # đổi chỗ hai viên: lẻ
    ([[0, 1, 2], [3, 4, 5]], None),         # tính bằng bảng rồi so sánh
]
for name, f in methods.items():
    for b, exp in edge:
        if exp is None:
            exp = table_lookup(b)
        assert f(b) == exp, (name, b, f(b), exp)
assert iddfs([[1, 2, 3], [5, 4, 0]]) == -1
print("Ví dụ + edge cases: OK")

# 12.2 Toàn bộ 720 trạng thái: mọi cách khớp BFS
for b in boards:
    ref = bfs(b)
    for name, f in methods.items():
        assert f(b) == ref, (name, b)
print("720 trạng thái: mọi cách khớp nhau")

# 12.3 IDDFS (baseline) trên mẫu ngẫu nhiên có seed
for b in sample:
    assert iddfs(b) == bfs(b), b
print("IDDFS khớp BFS trên 40 bảng ngẫu nhiên")

# 12.4 Phân bố khoảng cách
from collections import Counter
print(sorted(Counter(DIST_FROM_TARGET.values()).items()))

Ví dụ + edge cases: OK


720 trạng thái: mọi cách khớp nhau
IDDFS khớp BFS trên 40 bảng ngẫu nhiên
[(0, 1), (1, 2), (2, 3), (3, 5), (4, 6), (5, 7), (6, 10), (7, 12), (8, 12), (9, 16), (10, 23), (11, 25), (12, 28), (13, 39), (14, 44), (15, 40), (16, 29), (17, 21), (18, 18), (19, 12), (20, 6), (21, 1)]


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Đánh dấu `visited` khi **lấy ra** thay vì khi **đẩy vào**

**Code sai:**

In [23]:
def wrong_mark_on_pop(board):
    start = encode(board)
    seen = set()
    q = deque([(start, 0)])
    pushes = 0
    while q:
        s, d = q.popleft()
        if s == TARGET:
            return d, pushes
        if s in seen:
            continue
        seen.add(s)                              # SAI (kém hiệu quả): đánh dấu muộn
        z = s.index('0')
        for j in NEI[z]:
            t = swap(s, z, j)
            if t not in seen:
                q.append((t, d + 1)); pushes += 1
    return -1, pushes

print("số lần đẩy (đánh dấu muộn):", wrong_mark_on_pop([[3, 2, 4], [1, 5, 0]]))

def count_pushes_correct(board):
    start = encode(board); seen = {start}; q = deque([(start, 0)]); pushes = 0
    while q:
        s, d = q.popleft()
        if s == TARGET:
            return d, pushes
        z = s.index('0')
        for j in NEI[z]:
            t = swap(s, z, j)
            if t not in seen:
                seen.add(t); q.append((t, d + 1)); pushes += 1
    return -1, pushes

print("số lần đẩy (đánh dấu sớm):", count_pushes_correct([[3, 2, 4], [1, 5, 0]]))

số lần đẩy (đánh dấu muộn): (14, 272)
số lần đẩy (đánh dấu sớm): (14, 253)


**Vì sao sai:** kết quả vẫn đúng nhưng cùng một trạng thái bị đẩy vào hàng đợi **nhiều lần** → tốn thời gian và bộ nhớ (với bài lớn có thể TLE/MLE).

**Code đúng:** `seen.add(t)` ngay khi đẩy.

### Lỗi 2 — Láng giềng sai: coi chỉ số 2 và 3 là kề nhau (đi "xuyên dòng")

**Code sai:**

In [24]:
WRONG_NEI = [[1, 3], [0, 2, 4], [1, 3, 5], [0, 2, 4], [1, 3, 5], [2, 4]]   # SAI: 2<->3

def bfs_with(nei, board):
    start = encode(board)
    seen, q = {start}, deque([(start, 0)])
    while q:
        s, d = q.popleft()
        if s == TARGET:
            return d
        z = s.index('0')
        for j in nei[z]:
            t = swap(s, z, j)
            if t not in seen:
                seen.add(t); q.append((t, d + 1))
    return -1

b = [[1, 2, 0], [3, 4, 5]]
print("sai:", bfs_with(WRONG_NEI, b), " đúng:", bfs(b))

sai: 3  đúng: 13


**Vì sao sai:** khi duỗi bảng thành chuỗi, ô cuối hàng 0 (chỉ số 2) và ô đầu hàng 1 (chỉ số 3) **không kề nhau** trên bảng. Dùng `i ± 1` một cách máy móc sẽ sinh nước đi không hợp lệ.

**Code đúng:** sinh láng giềng từ toạ độ (`build_neighbors`) hoặc viết tay cẩn thận.

### Lỗi 3 — DFS thuần (không giới hạn độ sâu) trả về đường **không ngắn nhất**

**Code sai:**

In [25]:
import sys
def wrong_dfs(board):
    start = encode(board)
    seen = {start}
    def dfs(s, d):
        if s == TARGET:
            return d
        z = s.index('0')
        for j in NEI[z]:
            t = swap(s, z, j)
            if t not in seen:
                seen.add(t)
                r = dfs(t, d + 1)
                if r != -1:
                    return r                     # SAI: đường đầu tiên tìm thấy
        return -1
    return dfs(start, 0)

b = [[4, 1, 2], [5, 0, 3]]
print("DFS:", wrong_dfs(b), " BFS:", bfs(b))

DFS: 11  BFS: 5


**Vì sao sai:** DFS đi theo nhánh đầu tiên tới khi gặp đích, không đảm bảo ngắn nhất (có thể ra hàng trăm bước).

**Code đúng:** BFS, hoặc IDDFS (DFS với giới hạn độ sâu tăng dần).

### Lỗi 4 — A\* với heuristic tính cả ô trống hoặc "nhân đôi"

In [26]:
def h_overestimate(s):
    return 2 * manhattan(s)                      # SAI: không chấp nhận được

def astar_with(hfun, board):
    start = encode(board)
    g = {start: 0}
    heap = [(hfun(start), 0, start)]
    closed = set()
    while heap:
        f, d, s = heapq.heappop(heap)
        if s == TARGET:
            return d
        if s in closed:
            continue
        closed.add(s)
        z = s.index('0')
        for j in NEI[z]:
            t = swap(s, z, j)
            if t not in closed and d + 1 < g.get(t, 1 << 30):
                g[t] = d + 1
                heapq.heappush(heap, (d + 1 + hfun(t), d + 1, t))
    return -1

worse = [b for b in boards if bfs(b) != -1 and astar_with(h_overestimate, b) != bfs(b)]
print("số bảng mà heuristic x2 cho kết quả sai:", len(worse), "ví dụ:", worse[:1],
      astar_with(h_overestimate, worse[0]) if worse else None, bfs(worse[0]) if worse else None)

số bảng mà heuristic x2 cho kết quả sai: 53 ví dụ: [[[0, 1, 4], [2, 3, 5]]] 19 17


**Vì sao sai:** heuristic **ước lượng quá** số bước thật → A\* có thể chốt một đường dài hơn trước khi khám phá đường ngắn nhất.

**Code đúng:** `h` = tổng Manhattan **không tính ô trống** (chấp nhận được và nhất quán).

## 14. Mẹo & nhận dạng pattern

- **"Số bước ít nhất để biến trạng thái A thành B"**, mỗi bước chi phí 1 → **BFS trên không gian trạng thái**.
- **Trạng thái nhỏ** (vài nghìn) → mã hoá thành chuỗi/tuple/số nguyên, dùng `set`.
- **Không gian lớn** → BFS hai chiều, A\*, IDA\*.
- **Bài trượt gạch / hoán vị** → kiểm tra **tính chẵn lẻ** trước để loại trường hợp vô nghiệm.
- **Nhiều truy vấn trên cùng đồ thị vô hướng** → BFS một lần từ đích, tra bảng.
- Họ bài: *752. Open the Lock*, *127. Word Ladder*, *1091. Shortest Path in Binary Matrix*, *847. Shortest Path Visiting All Nodes*.

## 15. Biến thể

1. **Bảng 3×3 (8-puzzle)**: `9!/2 = 181440` trạng thái tới được — BFS vẫn chạy; điều kiện giải được: số nghịch thế chẵn (chiều rộng lẻ).
2. **Bảng 4×4 (15-puzzle)**: quá lớn cho BFS; dùng IDA\* + heuristic mạnh. Chiều rộng chẵn → điều kiện chẵn lẻ phải tính cả **hàng của ô trống**.
3. **Trả về dãy nước đi** → lưu `parent` (mục 8).
4. **Đích bất kỳ** → BFS từ đích mới; điều kiện chẵn lẻ so sánh với đích.
5. **Bảng R×C tổng quát** → `build_neighbors(R, C)`.

In [27]:
# Biến thể 5: lời giải tổng quát R x C bằng BFS (chạy cho 2x3 và 2x2)
def sliding_general(board):
    R, C = len(board), len(board[0])
    nei = build_neighbors(R, C)
    start = tuple(x for row in board for x in row)
    target = tuple(list(range(1, R * C)) + [0])
    seen, q = {start}, deque([(start, 0)])
    while q:
        s, d = q.popleft()
        if s == target:
            return d
        z = s.index(0)
        for j in nei[z]:
            lst = list(s); lst[z], lst[j] = lst[j], lst[z]; t = tuple(lst)
            if t not in seen:
                seen.add(t); q.append((t, d + 1))
    return -1

print(sliding_general([[4, 1, 2], [5, 0, 3]]), sliding_general([[0, 3], [2, 1]]), sliding_general([[2, 1], [3, 0]]))

5 6 -1


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. IDDFS | DFS giới hạn độ sâu tăng dần | `O(β^d)` | `O(d)` | Bộ nhớ rất hạn chế |
| 2. BFS | Duyệt theo lớp, `visited` | `O(V·N)` | `O(V·N)` | Mặc định, nộp bài |
| 3. BFS hai chiều | Mở rộng hai phía, gặp ở giữa | ≈ `β^(d/2)` | như trên | Không gian lớn, biết đích |
| 4. A\* | `f = g + h`, Manhattan | ít hơn BFS thực tế | `O(V·N)` | Không gian lớn, có heuristic tốt |
| 5. Tra bảng | BFS một lần từ đích | `O(V·N)` + `O(N)` | `O(V·N)` | Nhiều truy vấn |

## 17. Bài tập

**Bài 1 (đoán output).** `slidingPuzzle([[1,2,3],[0,4,5]])` và `slidingPuzzle([[5,4,3],[2,1,0]])` = ?

**Bài 2 (tính tay).** Tính số nghịch thế của `"412503"` và `"123540"`. Trạng thái nào giải được?

**Bài 3 (điền TODO).** Hoàn thiện `solve_path(board)` trả về **danh sách trạng thái** trên một đường đi ngắn nhất (hoặc `[]`).

**Bài 4 (tự viết lại).** Viết lại BFS dùng **số nguyên** để mã hoá trạng thái (mỗi ô 3 bit) thay vì chuỗi.

**Bài 5 (mini challenge).** Đếm có bao nhiêu trạng thái cần **đúng** `k` bước, với mọi `k`, và tìm trạng thái "khó nhất".

In [28]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def solve_path(board):
    # TODO: BFS lưu parent; truy ngược từ TARGET
    pass

# Bài 4 — TODO
def bfs_int(board):
    # TODO: code = sum(v << (3 * i)); lấy ô i: (code >> 3*i) & 7; đổi chỗ bằng phép bit
    pass

# Bài 5 — TODO
def hardest_states():
    # TODO: BFS từ đích, gom theo khoảng cách
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `"123045"`: `0` ở chỉ số 3.
- Mức 2: trượt `4` rồi `5` sang trái → 2 bước.
- Mức 3: `[[5,4,3],[2,1,0]]`: nghịch thế của `54321` = 10 (chẵn) → giải được; BFS cho `14` bước (khá xa, nhưng chưa phải xa nhất — xa nhất là 21).

**Bài 2.**
- Mức 1: `"412503"` → dãy `4,1,2,5,3`.
- Mức 2: nghịch thế: (4,1),(4,2),(4,3),(5,3) = 4 → chẵn → giải được.
- Mức 3: `"123540"` → `1,2,3,5,4`: 1 nghịch thế → lẻ → `-1`.

**Bài 3.**
- Mức 1: `parent[start] = None`.
- Mức 2: khi đẩy `t`, ghi `parent[t] = s`.
- Mức 3: truy ngược từ đích, đảo danh sách.

**Bài 4.**
- Mức 1: 6 ô × 3 bit = 18 bit.
- Mức 2: vị trí ô trống lưu riêng cùng trạng thái để khỏi tìm.
- Mức 3: đổi chỗ: lấy giá trị `v` ở ô `j`, xoá ô `j`, đặt `v` vào ô `z`.

**Bài 5.**
- Mức 1: dùng `DIST_FROM_TARGET`.
- Mức 2: `Counter(dist.values())`.
- Mức 3: trạng thái có khoảng cách lớn nhất (21).

## 19. Đáp án tham khảo

In [29]:
# Bài 1
assert Solution().slidingPuzzle([[1, 2, 3], [0, 4, 5]]) == 2
ans_b1 = Solution().slidingPuzzle([[5, 4, 3], [2, 1, 0]])
print("[[5,4,3],[2,1,0]] ->", ans_b1)
assert ans_b1 == bfs([[5, 4, 3], [2, 1, 0]]) == 14

# Bài 2
assert inversions("412503") == 4 and inversions("123540") == 1
assert Solution().slidingPuzzle([[1, 2, 3], [5, 4, 0]]) == -1

# Bài 3
def solve_path_ans(board):
    start = encode(board)
    parent = {start: None}
    q = deque([start])
    while q:
        s = q.popleft()
        if s == TARGET:
            path = []
            while s:
                path.append(s); s = parent[s]
            return path[::-1]
        z = s.index('0')
        for j in NEI[z]:
            t = swap(s, z, j)
            if t not in parent:
                parent[t] = s; q.append(t)
    return []

p = solve_path_ans([[4, 1, 2], [5, 0, 3]])
assert len(p) - 1 == 5 and p[0] == "412503" and p[-1] == TARGET
assert all(t in next_states(s) for s, t in zip(p, p[1:]))
assert solve_path_ans([[1, 2, 3], [5, 4, 0]]) == []

# Bài 4
def bfs_int_ans(board):
    vals = [x for row in board for x in row]
    code = sum(v << (3 * i) for i, v in enumerate(vals))
    target = sum(v << (3 * i) for i, v in enumerate([1, 2, 3, 4, 5, 0]))
    z0 = vals.index(0)
    seen = {code}
    q = deque([(code, z0, 0)])
    while q:
        c, z, d = q.popleft()
        if c == target:
            return d
        for j in NEI[z]:
            v = (c >> (3 * j)) & 7
            nc = c & ~(7 << (3 * j)) | (v << (3 * z))   # ô z đang là 0
            if nc not in seen:
                seen.add(nc); q.append((nc, j, d + 1))
    return -1

for b in random.sample(boards, 100):
    assert bfs_int_ans(b) == bfs(b)

# Bài 5
def hardest_states_ans():
    cnt = Counter(DIST_FROM_TARGET.values())
    mx = max(cnt)
    return dict(sorted(cnt.items())), [s for s, d in DIST_FROM_TARGET.items() if d == mx]

dist_count, hardest = hardest_states_ans()
assert sum(dist_count.values()) == 360 and max(dist_count) == 21
print("khó nhất (21 bước):", hardest)
print("Tất cả đáp án đúng")

[[5,4,3],[2,1,0]] -> 14
khó nhất (21 bước): ['450123']
Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template BFS trạng thái:**

```python
nei = [[1,3],[0,2,4],[1,5],[0,4],[1,3,5],[2,4]]
start, target = encode(board), "123450"
seen, q = {start}, deque([(start, 0)])
while q:
    s, d = q.popleft()
    if s == target: return d
    z = s.index('0')
    for j in nei[z]:
        t = swap(s, z, j)
        if t not in seen: seen.add(t); q.append((t, d + 1))
return -1
```

**Checklist trước khi submit:**

- [ ] Bảng láng giềng đúng (2 và 3 **không** kề nhau).
- [ ] Đánh dấu `seen` khi **đẩy vào** hàng đợi.
- [ ] Trường hợp đã ở đích → `0`.
- [ ] Vô nghiệm → `-1` (BFS cạn hoặc kiểm tra chẵn lẻ).
- [ ] Dùng BFS (không phải DFS thuần) cho đường ngắn nhất.
- [ ] A\*: heuristic không tính ô trống.
- [ ] Tên method đúng: `slidingPuzzle(self, board)`.